#### Combine All Cleaned Datasets

Joins all tables in `cleaned_data/` on `(participant_id, timepoint)`. Demographics from `participants_cleaned` are broadcast via `participant_id`. All joins use `how='outer'` — missing values become NaN.

In [1]:
import os
import pandas as pd
from IPython.display import display

CLEANED_DATA_DIR = 'cleaned_data'
OUTPUT_DIR = 'merged_data'
os.makedirs(OUTPUT_DIR, exist_ok=True)

#### 1. Participants — no timepoint, demographics broadcast via `participant_id`

In [2]:
participants = pd.read_csv(os.path.join(CLEANED_DATA_DIR, 'participants_cleaned.csv'))
print(f"Participants shape: {participants.shape}")
display(participants.head(3))

Participants shape: (3960, 5)


,participant_id,biological_sex,race,geolocation,age
0,SDY269.SUB112836,Female,White,Georgia,28.0
1,SDY269.SUB112849,Female,Black,Georgia,39.0
2,SDY269.SUB112854,Male,Black,Georgia,46.0


#### 2. HAI — pivot `virus_strain` to columns

In [3]:
hai = pd.read_csv(os.path.join(CLEANED_DATA_DIR, 'hai_cleaned.csv'))
print(f"HAI shape: {hai.shape}")
print(f"Unique virus strains: {hai['virus_strain'].unique()}")

hai_pivot = hai.pivot_table(
    index=['participant_id', 'timepoint'],
    columns='virus_strain',
    values='value',
    aggfunc='mean'
).reset_index()
hai_pivot.columns.name = None
# Prefix columns to avoid name collisions in the final table
hai_pivot.rename(columns={
    c: f'hai_{c}' for c in hai_pivot.columns if c not in ['participant_id', 'timepoint']
}, inplace=True)

# Drop the 'hai_-' column if present — it comes from rows where virus_strain is literally "-"
if 'hai_-' in hai_pivot.columns:
    print("Dropping 'hai_-' column (virus_strain='-' is a data artifact)")
    hai_pivot.drop(columns=['hai_-'], inplace=True)

print(f"\nHAI pivoted shape: {hai_pivot.shape}")
display(hai_pivot.head(3))

HAI shape: (128177, 5)
Unique virus strains: <StringArray>
[        'H1N1 A/South Carolina/1/1918',
                'H1N1 A/Weiss/JY2/1943',
          'H1N1 A/Fort Monmouth/1/1947',
                 'H1N1 A/Denver/1/1957',
             'H1N1 A/New Jersey/8/1976',
                  'H1N1 A/USSR/90/1977',
                'H1N1 A/Brazil/11/1978',
                  'H1N1 A/Chile/1/1983',
              'H1N1 A/Singapore/6/1986',
                 'H1N1 A/Texas/36/1991',
              'H1N1 A/Beijing/262/1995',
         'H1N1 A/New Caledonia/20/1999',
        'H1N1 A/Solomon Islands/3/2006',
              'H1N1 A/Brisbane/59/2007',
             'H1N1 A/California/7/2009',
              'H1N1 A/Michigan/45/2015',
              'H3N2 A/Hong Kong/1/1968',
          'H3N2 A/Port Chalmers/1/1973',
                  'H3N2 A/Texas/1/1977',
            'H3N2 A/Mississippi/1/1985',
               'H3N2 A/Sichuan/60/1989',
               'H3N2 A/Shandong/9/1993',
             'H3N2 A/Nanchang/933/1995'

,participant_id,timepoint,hai_Anc B/Lee/1940,hai_Anc B/Maryland/1959,hai_Anc B/Singapore/1964,hai_H1N1 A/Beijing/262/1995,hai_H1N1 A/Brazil/11/1978,hai_H1N1 A/Brisbane/2/2018,hai_H1N1 A/Brisbane/59/2007,hai_H1N1 A/California/7/2009,...,hai_Vic B/Washington/2/2019,hai_Yam B/Brisbane/3/2007,hai_Yam B/Florida/4/2006,hai_Yam B/Harbin/7/1994,hai_Yam B/Massachusetts/2/2012,hai_Yam B/Phuket/3073/2013,hai_Yam B/Sichuan/379/1999,hai_Yam B/Texas/6/2011,hai_Yam B/Wisconsin/1/2010,hai_Yam B/Yamagata/16/1988
0,2016_UGA.ID_001,0.0,NaN,NaN,NaN,40.0,5.0,NaN,10.0,160.0,...,NaN,NaN,640.0,640.0,640.0,320.0,640.0,160.0,320.0,160.0
1,2016_UGA.ID_001,28.0,NaN,NaN,NaN,40.0,5.0,NaN,40.0,320.0,...,NaN,NaN,640.0,1280.0,1280.0,1280.0,1280.0,320.0,320.0,320.0
2,2016_UGA.ID_002,0.0,NaN,NaN,NaN,80.0,5.0,NaN,40.0,20.0,...,NaN,NaN,80.0,160.0,80.0,80.0,160.0,40.0,80.0,40.0


#### 3. Flow Cytometry — pivot cell population `name` to columns

In [4]:
flow = pd.read_csv(os.path.join(CLEANED_DATA_DIR, 'flow_cleaned.csv'), low_memory=False)
print(f"Flow shape: {flow.shape}")
print(f"Unique cell population names: {flow['name'].nunique()}")
print(f"Sample names: {flow['name'].unique()[:8]}")

flow_pivot = flow.pivot_table(
    index=['participant_id', 'timepoint'],
    columns='name',
    values='value',
    aggfunc='mean'
).reset_index()
flow_pivot.columns.name = None

def clean_col_name(name):
    """Sanitize cell population names for use as column names."""
    return (name.replace(' ', '_')
                .replace('+', 'pos')
                .replace('-', 'neg')
                .replace('/', '_')
                .replace('%', 'pct')
                .replace('(', '')
                .replace(')', ''))

flow_pivot.rename(columns={
    c: f'flow_{clean_col_name(c)}' for c in flow_pivot.columns if c not in ['participant_id', 'timepoint']
}, inplace=True)

print(f"\nFlow pivoted shape: {flow_pivot.shape}")
display(flow_pivot.iloc[:3, :6])

Flow shape: (213624, 8)
Unique cell population names: 28
Sample names: <StringArray>
[                                  'Live cd45+ cells',
                                      'Naive b cells',
 'Memory class switched + non-class switched b cells',
                      'Memory class switched b cells',
                                            'B cells',
                                                  nan,
                             'Ab-secreting cells asc',
                                        'Neutrophils']
Length: 8, dtype: str

Flow pivoted shape: (1371, 30)


,participant_id,timepoint,flow_Abnegsecreting_cells_asc,flow_B_cells,flow_Cd4_t_cells,flow_Cd4_tcn
0,SDY180.SUB119262,-7.0,0.207948,37.06170,343.651313,99.39500
1,SDY180.SUB119262,0.0,0.251182,38.15872,197.608956,85.42314
2,SDY180.SUB119262,0.5,0.365098,53.41014,269.133436,100.62780


#### 4. Microarray — pivot `gene_symbol` to columns (mean across platforms)

In [5]:
array = pd.read_csv(os.path.join(CLEANED_DATA_DIR, 'array_cleaned.csv'))
print(f"Array shape: {array.shape}")
print(f"Unique genes: {array['gene_symbol'].nunique()}")
print(f"Platforms: {array['platform_id'].unique()}")

array_pivot = array.pivot_table(
    index=['participant_id', 'timepoint'],
    columns='gene_symbol',
    values='value',
    aggfunc='mean'  # average across platforms when a gene appears on multiple
).reset_index()
array_pivot.columns.name = None
array_pivot.rename(columns={
    c: f'array_{c}' for c in array_pivot.columns if c not in ['participant_id', 'timepoint']
}, inplace=True)

print(f"\nArray pivoted shape: {array_pivot.shape}")
display(array_pivot.iloc[:3, :6])

Array shape: (16112494, 6)
Unique genes: 39665
Platforms: <StringArray>
['GPL570', 'GPL13158', 'GPL6947', 'GPL10558']
Length: 4, dtype: str

Array pivoted shape: (683, 39667)


,participant_id,timepoint,array_7A5,array_A1BG,array_A1BG-AS1,array_A1CF
0,SDY1119.SUB179642,0,NaN,3.708722,NaN,3.268319
1,SDY1119.SUB179642,7,NaN,4.125560,NaN,3.233171
2,SDY1119.SUB179643,0,NaN,4.441810,NaN,3.162109


#### 5. BCR — aggregate numeric columns (mean) per `(participant_id, timepoint)` + sequence count

In [6]:
bcr = pd.read_csv(os.path.join(CLEANED_DATA_DIR, 'bcr_cleaned.csv'), low_memory=False)
print(f"BCR shape: {bcr.shape}")

# Identify numeric columns excluding the keys and any duplicated demographic columns
# (sex, age, biological_sex are already in participants_cleaned)
exclude = {'participant_id', 'timepoint', 'sex', 'age', 'biological_sex',
           'subject_id', 'sample_id', 'clone_id'}
bcr_numeric_cols = [
    c for c in bcr.select_dtypes(include='number').columns if c not in exclude
]
print(f"Numeric BCR columns to aggregate ({len(bcr_numeric_cols)}): {bcr_numeric_cols}")

bcr_agg = (bcr
           .groupby(['participant_id', 'timepoint'])[bcr_numeric_cols]
           .mean()
           .reset_index())

bcr_counts = (bcr
              .groupby(['participant_id', 'timepoint'])
              .size()
              .reset_index(name='sequence_count'))

bcr_agg = bcr_agg.merge(bcr_counts, on=['participant_id', 'timepoint'])

# Prefix all feature columns
bcr_agg.rename(columns={
    c: f'bcr_{c}' for c in bcr_agg.columns if c not in ['participant_id', 'timepoint']
}, inplace=True)

print(f"\nBCR aggregated shape: {bcr_agg.shape}")
display(bcr_agg.head(3))

BCR shape: (252479, 66)
Numeric BCR columns to aggregate (31): ['v_score', 'v_identity', 'v_support', 'd_score', 'd_identity', 'd_support', 'j_score', 'j_identity', 'j_support', 'v_sequence_start', 'v_sequence_end', 'v_germline_end', 'd_sequence_start', 'd_sequence_end', 'd_germline_start', 'd_germline_end', 'j_sequence_start', 'j_sequence_end', 'j_germline_start', 'j_germline_end', 'junction_length', 'np1_length', 'np2_length', 'consensus_count', 'duplicate_count', 'seq_len', 'collapse_count', 'v_germline_length', 'd_germline_length', 'j_germline_length', 'mu_freq']

BCR aggregated shape: (20, 34)


,participant_id,timepoint,bcr_v_score,bcr_v_identity,bcr_v_support,bcr_d_score,bcr_d_identity,bcr_d_support,bcr_j_score,bcr_j_identity,...,bcr_np2_length,bcr_consensus_count,bcr_duplicate_count,bcr_seq_len,bcr_collapse_count,bcr_v_germline_length,bcr_d_germline_length,bcr_j_germline_length,bcr_mu_freq,bcr_sequence_count
0,EXT100.321P04,0,395.223134,0.944241,1.861158e-25,21.258491,0.973867,30.184848,79.257219,0.967854,...,7.231243,3.748753,2.378289,390.060921,2.879523,318.546024,11.949314,45.191563,0.051260,19041
1,EXT100.321P04,5,391.666607,0.925600,7.817005e-41,19.986196,0.962713,35.001172,76.988281,0.958193,...,7.629487,3.581430,3.478151,391.008691,1.852485,318.279047,11.862208,45.421690,0.069617,16683
2,EXT100.321P04,7,398.246284,0.941665,2.120378e-32,19.023650,0.952953,26.556519,63.185070,0.967990,...,7.338920,3.436753,2.402871,373.106383,2.085908,318.497238,11.607076,44.839779,0.003051,13724


#### 6. Transcriptomics — concat 4 datasets (outer join aligns gene columns, fills missing with NaN)

In [7]:
tx_files = {
    '2019_UGA': 'transcriptomics_2019_UGA_cleaned.csv',
    '2020_UGA': 'transcriptomics_2020_UGA_cleaned.csv',
    'SDY2867':  'transcriptomics_SDY2867_cleaned.csv',
    'SDY2941':  'transcriptomics_SDY2941_cleaned.csv',
}

tx_dfs = []
for label, fname in tx_files.items():
    path = os.path.join(CLEANED_DATA_DIR, fname)
    df = pd.read_csv(path)
    gene_cols = [c for c in df.columns if c not in ('participant_id', 'timepoint')]
    print(f"{label}: {df.shape[0]} rows × {len(gene_cols)} gene columns")
    tx_dfs.append(df)

# Outer concat aligns on shared column names; missing genes become NaN
transcriptomics = pd.concat(tx_dfs, axis=0, join='outer').reset_index(drop=True)
gene_cols_all = [c for c in transcriptomics.columns if c not in ('participant_id', 'timepoint')]
print(f"\nCombined transcriptomics: {transcriptomics.shape[0]} rows × {len(gene_cols_all)} gene columns")
display(transcriptomics[['participant_id', 'timepoint']].head(3))

2019_UGA: 764 rows × 42531 gene columns
2020_UGA: 96 rows × 56414 gene columns
SDY2867: 361 rows × 54902 gene columns
SDY2941: 316 rows × 54902 gene columns

Combined transcriptomics: 1537 rows × 79224 gene columns


,participant_id,timepoint
0,2019_UGA.ID_001,0
1,2019_UGA.ID_001,3
2,2019_UGA.ID_001,7


#### 7. Merge — union all `(participant_id, timepoint)` pairs, then outer-join each table

In [8]:
timepoint_tables = {
    'HAI':             hai_pivot,
    'Flow':            flow_pivot,
    'Array':           array_pivot,
    'BCR':             bcr_agg,
    'Transcriptomics': transcriptomics,
}

# Step 1: union of all (participant_id, timepoint) pairs
all_pairs = pd.concat(
    [t[['participant_id', 'timepoint']] for t in timepoint_tables.values()],
    ignore_index=True
).drop_duplicates().reset_index(drop=True)
print(f"Total unique (participant_id, timepoint) pairs: {len(all_pairs)}")

# Step 2: broadcast participant demographics to every timepoint row
merged = all_pairs.merge(participants, on='participant_id', how='left')
print(f"After joining participants:   {merged.shape}")

# Step 3: outer-join each feature table
for name, table in timepoint_tables.items():
    merged = merged.merge(table, on=['participant_id', 'timepoint'], how='outer')
    print(f"After joining {name:<15}: {merged.shape}")

print(f"\nFinal combined dataset: {merged.shape[0]:,} rows × {merged.shape[1]:,} columns")

Total unique (participant_id, timepoint) pairs: 11504
After joining participants:   (11504, 6)
After joining HAI            : (11504, 74)
After joining Flow           : (11504, 102)
After joining Array          : (11504, 39767)
After joining BCR            : (11504, 39799)
After joining Transcriptomics: (11504, 119023)

Final combined dataset: 11,504 rows × 119,023 columns


#### 8. Column Summary

In [9]:
col_groups = {
    'Keys':            [c for c in ['participant_id', 'timepoint'] if c in merged.columns],
    'Demographics':    [c for c in participants.columns if c != 'participant_id'],
    'HAI':             [c for c in merged.columns if c.startswith('hai_')],
    'Flow':            [c for c in merged.columns if c.startswith('flow_')],
    'Array':           [c for c in merged.columns if c.startswith('array_')],
    'BCR':             [c for c in merged.columns if c.startswith('bcr_')],
    'Transcriptomics': [c for c in merged.columns if c.startswith('ENSG')],
}

print("Column counts by group:")
for group, cols in col_groups.items():
    print(f"  {group:<18}: {len(cols):>6}")
print(f"  {'TOTAL':<18}: {sum(len(v) for v in col_groups.values()):>6}")

print("\nSample rows (key + demographic columns):")
preview_cols = col_groups['Keys'] + col_groups['Demographics']
display(merged[preview_cols].head(5))

Column counts by group:
  Keys              :      2
  Demographics      :      4
  HAI               :     68
  Flow              :     28
  Array             :  39665
  BCR               :     32
  Transcriptomics   :  79224
  TOTAL             : 119023

Sample rows (key + demographic columns):


,participant_id,timepoint,biological_sex,race,geolocation,age
0,2016_UGA.ID_001,0.0,Female,Unknown,Georgia,29.0
1,2016_UGA.ID_001,28.0,Female,Unknown,Georgia,29.0
2,2016_UGA.ID_002,0.0,Female,Unknown,Georgia,29.0
3,2016_UGA.ID_002,28.0,Female,Unknown,Georgia,29.0
4,2016_UGA.ID_003,0.0,Female,Unknown,Georgia,28.0


#### 9. Save

In [10]:
# output_path = os.path.join(OUTPUT_DIR, 'combined_dataset.csv')
# merged.to_csv(output_path, index=False)
# print(f"Saved: {output_path}")
# print(f"Shape: {merged.shape[0]:,} rows × {merged.shape[1]:,} columns")

In [11]:
merged.head()

,participant_id,timepoint,biological_sex,race,geolocation,age,hai_Anc B/Lee/1940,hai_Anc B/Maryland/1959,hai_Anc B/Singapore/1964,hai_H1N1 A/Beijing/262/1995,...,ENSG00000310527,ENSG00000310529,ENSG00000310530,ENSG00000310532,ENSG00000310533,ENSG00000310534,ENSG00000310535,ENSG00000310536,ENSG00000310537,ENSG00000310539
0,2016_UGA.ID_001,0.0,Female,Unknown,Georgia,29.0,NaN,NaN,NaN,40.0,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
1,2016_UGA.ID_001,28.0,Female,Unknown,Georgia,29.0,NaN,NaN,NaN,40.0,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
2,2016_UGA.ID_002,0.0,Female,Unknown,Georgia,29.0,NaN,NaN,NaN,80.0,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
3,2016_UGA.ID_002,28.0,Female,Unknown,Georgia,29.0,NaN,NaN,NaN,40.0,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
4,2016_UGA.ID_003,0.0,Female,Unknown,Georgia,28.0,NaN,NaN,NaN,80.0,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
